In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Φόρτωση Δεδομένων 
train_path = 'processed/train_final_scaled.csv'
test_path = 'processed/test_final_scaled.csv'

# Διαβάζουμε τα columns και φτιάχνουμε τα types
columns = pd.read_csv(train_path, nrows=0).columns.tolist()
features = [col for col in columns if col != "Label"]
dtypes = {col: "float32" for col in features}
dtypes["Label"] = "int16"

train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

# Διαχωρισμός σε Features (X)
X_train = train_df[features].to_numpy(dtype=np.float32)
X_test = test_df[features].to_numpy(dtype=np.float32)

# Μετατροπή σε Labels (0=Benign, 1=Attack)
BENIGN_LABEL = 1

# Ορίζουμε απευθείας τα binary labels (ό,τι δεν είναι 1, γίνεται 1, δηλαδή Attack)
y_train_bin = (train_df["Label"].to_numpy() != BENIGN_LABEL).astype(int)
y_test_bin = (test_df["Label"].to_numpy() != BENIGN_LABEL).astype(int)

In [ ]:
# --- SUPERVISED MODEL: Random Forest ---
print("\n--- ΑΠΟΤΕΛΕΣΜΑΤΑ SUPERVISED (Random Forest) ---")
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1) 
rf.fit(X_train, y_train_bin)
rf_preds = rf.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test_bin, rf_preds):.4f}")
print(f"Precision: {precision_score(y_test_bin, rf_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test_bin, rf_preds, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test_bin, rf_preds, zero_division=0):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_bin, rf_preds, labels=[0, 1]))

In [ ]:
# --- UNSUPERVISED MODEL: K-Means ---
print("\n--- ΑΠΟΤΕΛΕΣΜΑΤΑ UNSUPERVISED (K-Means) ---")
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(X_train)
kmeans_preds = kmeans.predict(X_test)

# Διόρθωση αν το K-Means έβαλε ανάποδα τα labels 0 και 1
if accuracy_score(y_test_bin, kmeans_preds) < 0.5:
    kmeans_preds = 1 - kmeans_preds

print(f"Accuracy:  {accuracy_score(y_test_bin, kmeans_preds):.4f}")
print(f"Precision: {precision_score(y_test_bin, kmeans_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test_bin, kmeans_preds, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test_bin, kmeans_preds, zero_division=0):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_bin, kmeans_preds, labels=[0, 1]))